# Template Eksperimen MSML: Pra-Pemrosesan Data & Analisis Risiko Diabetes
### Proyek Akhir: Membangun Sistem Machine Learning — Dicoding

**Nama**  : Riswandi  
**Proyek**: Pemodelan Prediksi Diagnosis Diabetes Menggunakan MLOps & MLflow  

---

## 1. Perkenalan Dataset (Dataset Introduction)

### 📌 Sumber Dataset
- **Nama Dataset**: Pima Indians Diabetes Database
- **Sumber / Repositori**: [Kaggle — Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
- **Penyedia Data original**: *National Institute of Diabetes and Digestive and Kidney Diseases* (UCI Machine Learning Repository)

### 📊 Gambaran Singkat Isi Data
Dataset ini bertujuan untuk memprediksi secara medis apakah seorang pasien menderita penyakit diabetes berdasarkan pengukuran diagnostik tertentu. Seluruh pasien dalam dataset ini adalah perempuan berusia minimal 21 tahun berketurunan Pima Indian.

#### Rincian Fitur dan Variabel:
1. **`Pregnancies`**: Jumlah kali hamil (integer)
2. **`Glucose`**: Konsentrasi glukosa plasma selama 2 jam dalam tes toleransi glukosa oral (mg/dL)
3. **`BloodPressure`**: Tekanan darah diastolik (mm Hg)
4. **`SkinThickness`**: Ketebalan lipatan kulit triceps (mm)
5. **`Insulin`**: Serum insulin 2 jam (mu U/ml)
6. **`BMI`**: Indeks massa tubuh (berat kg / (tinggi m)^2)
7. **`DiabetesPedigreeFunction`**: Fungsi silsilah diabetes (skor riwayat genetik keluarga)
8. **`Age`**: Usia pasien (tahun)
9. **`Outcome`** *(Target)*: Variabel biner klasifikasi (0: Tidak Diabetes, 1: Diabetes)

---
## 2. Memuat dan Memeriksa Data Mentah (`diabetes_raw.csv`)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Simpan dan baca data mentah (diabetes_raw.csv)
PATH_RAW = 'diabetes_raw.csv'
if not os.path.exists(PATH_RAW):
    PATH_RAW = '../diabetes_raw.csv'

df_raw = pd.read_csv(PATH_RAW)
print(f'Dataset Mentah Dimuat: {df_raw.shape[0]} baris × {df_raw.shape[1]} kolom')
print('\nInformasi Tipe Data & Missing Values:')
df_raw.info()

Dataset Mentah Dimuat: 768 baris × 9 kolom

Informasi Tipe Data & Missing Values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [2]:
# Lima baris pertama data mentah
df_raw.head(3)

Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
6,148,72,35,0,33.6,0.627,50,1
1,85,66,29,0,26.6,0.351,31,0
8,183,64,0,0,23.3,0.672,32,1


---
## 3. Analisis Data Eksploratori (EDA) & Imputasi Nilai Tidak Logis

Beberapa atribut medis seperti `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, dan `BMI` memiliki nilai `0` yang tidak mungkin secara medis. Nilai `0` tersebut diubah menjadi `NaN` lalu diimputasi dengan nilai median sesuai kelompok kelas target.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Distribusi kelas Outcome
counts = df_raw['Outcome'].value_counts()
axes[0].bar(['0: Non-Diabetes', '1: Diabetes'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Distribusi Kelas Target Outcome', fontweight='bold')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, f'{v} ({v/len(df_raw)*100:.1f}%)', ha='center', fontweight='bold')

# Heatmap Korelasi
sns.heatmap(df_raw.corr(), annot=True, fmt='.2f', cmap='YlGnBu', ax=axes[1], cbar=False)
axes[1].set_title('Matriks Korelasi Fitur Medis', fontweight='bold')

plt.tight_layout()
plt.show()

In [4]:
kolom_medis = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('Jumlah nilai 0 tidak logis per atribut:')
for col in kolom_medis:
    n_zero = (df_raw[col] == 0).sum()
    print(f'  {col:<14}: {n_zero} sampel')

# Ganti 0 dengan NaN
df_clean = df_raw.copy()
df_clean[kolom_medis] = df_clean[kolom_medis].replace(0, np.nan)

# Imputasi dengan median per kelompok Outcome
for col in kolom_medis:
    df_clean[col] = df_clean.groupby('Outcome')[col].transform(lambda x: x.fillna(x.median()))

print('\n✅ Imputasi nilai median selesai. Tidak ada nilai kosong tersisa.')

Jumlah nilai 0 tidak logis per atribut:
  Glucose       : 5 sampel
  BloodPressure : 35 sampel
  SkinThickness : 227 sampel
  Insulin       : 374 sampel
  BMI           : 11 sampel

✅ Imputasi nilai median selesai. Tidak ada nilai kosong tersisa.


---
## 4. Pembagian Dataset & Standarisasi Fitur

Dataset dibagi menjadi **80% data latih (614 sampel)** dan **20% data uji (154 sampel)** dengan metode Stratified Split. Fitur dinormalisasi menggunakan `StandardScaler`.

In [5]:
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']

# Split 80/20 Stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standard Scaling
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# Simpan berkas CSV hasil pra-pemrosesan
pd.DataFrame(X_tr_sc, columns=X.columns).to_csv('X_train.csv', index=False)
pd.DataFrame(X_te_sc, columns=X.columns).to_csv('X_test.csv', index=False)
pd.DataFrame(y_train).to_csv('y_train.csv', index=False)
pd.DataFrame(y_test).to_csv('y_test.csv', index=False)

print('Pembagian Dataset Terstratifikasi:')
print(f'  Data Latih (X_train) : {X_train.shape[0]} sampel × {X_train.shape[1]} fitur')
print(f'  Data Uji (X_test)   : {X_test.shape[0]} sampel × {X_test.shape[1]} fitur')
print('\nBerkas tersimpan:')
print('  ✓ X_train.csv & X_test.csv (fitur ter-skala)')
print('  ✓ y_train.csv & y_test.csv (label target)')
print('\n✅ Pra-pemrosesan data selesai 100%!')

Pembagian Dataset Terstratifikasi:
  Data Latih (X_train) : 614 sampel × 8 fitur
  Data Uji (X_test)   : 154 sampel × 8 fitur

Berkas tersimpan:
  ✓ X_train.csv & X_test.csv (fitur ter-skala)
  ✓ y_train.csv & y_test.csv (label target)

✅ Pra-pemrosesan data selesai 100%!
